In [2]:
import pandas as pd
import sqlite3
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('amazon_cleaned.csv')
df['date'] = pd.to_datetime(df['date'])

conn = sqlite3.connect('amazon sales.db')

df.to_sql('sales', conn, if_exists='replace', index=False)
print("Data loaded into SQLite")
print("Total rows in DB: ", pd.read_sql("SELECT COUNT(*) as total FROM sales",conn).iloc[0,0])

Data loaded into SQLite
Total rows in DB:  121177


In [8]:
q1 = pd.read_sql("""
    SELECT
        COUNT(*)                 AS total_orders,
        ROUND(SUM(revenue), 2)   AS total_revenue,
        ROUND(AVG(amount), 2)    AS avg_order_value,
        ROUND(MAX(amount), 2)    AS max_order_value,
        ROUND(MIN(amount), 2)    AS min_order_value
    FROM sales
""", conn)
print("\Q1:nOverall SUMMARY")
print(q1)

\Q1:nOverall SUMMARY
   total_orders  total_revenue  avg_order_value  max_order_value  \
0        121177     76031771.0           648.56           5584.0   

   min_order_value  
0              0.0  


In [4]:
q2 = pd.read_sql("""
    SELECT
        category,
        COUNT(*)                  AS total_orders,
        ROUND(SUM(revenue),2)     AS total_revenue,
        ROUND(AVG(amount), 2)     AS avg_order_value
    FROM sales
    GROUP BY category
    ORDER BY total_revenue DESC
""", conn)
print("\n q2: revenue by category === ")
print(q2)


 q2: revenue by category === 
        category  total_orders  total_revenue  avg_order_value
0            Set         47040     37932332.0           833.38
1          kurta         46716     20674816.0           455.93
2  Western Dress         14704     10707932.0           762.79
3            Top         10165      5242931.0           526.10
4   Ethnic Dress          1093       762949.0           723.90
5         Blouse           881       441259.0           520.33
6         Bottom           420       142870.0           358.73
7          Saree           155       125767.0           799.57
8        Dupatta             3          915.0           305.00


In [7]:
q3 = pd.read_sql(""" 
    SELECT
        status,
        COUNT(*)                      AS total_orders,
        ROUND(100.0 * COUNT(*) /
            SUM(COUNT(*)) OVER(),1)   AS percentage
    FROM sales
    GROUP BY STATUS
    ORDER BY total_orders DESC
""", conn)
print("\n=== Q3: ORDER STATUS BREAKDOWN ===")
print(q3)


=== Q3: ORDER STATUS BREAKDOWN ===
                           status  total_orders  percentage
0                         Shipped         77593        64.0
1    Shipped - Delivered to Buyer         28761        23.7
2                       Cancelled         10766         8.9
3    Shipped - Returned to Seller          1950         1.6
4             Shipped - Picked Up           973         0.8
5                         Pending           656         0.5
6   Pending - Waiting for Pick Up           281         0.2
7   Shipped - Returning to Seller           145         0.1
8      Shipped - Out for Delivery            35         0.0
9     Shipped - Rejected by Buyer            11         0.0
10      Shipped - Lost in Transit             5         0.0
11              Shipped - Damaged             1         0.0


In [14]:
# q4 = pd.read_sql("""
#     SELECT
#         ship_state,
#         COUNT(*)                 AS total_orders,
#         ROUND(SUM(revenue), 2)   AS total_reveune
#     FROM sales
#     GROUP BY ship_state
#     ORDER BY total_revenue DESC
#     LIMIT 10
# """, conn)
# print("\n=== Q4: TOP 10 STATES BY REVENUE ===")
# print(q4)

In [16]:
q5 = pd.read_sql("""
    SELECT
        month,
        COUNT(*)                  AS total_orders,
        ROUND(SUM(revenue), 2)   as total_revenue
    FROM sales
    GROUP BY month
    ORDER BY month
""", conn)
print("\n=== Q5: MONTHLY REVENUE TREND ===")
print(q5)


=== Q5: MONTHLY REVENUE TREND ===
     month  total_orders  total_revenue
0  2022-03           162        98261.0
1  2022-04         46068     27847245.0
2  2022-05         39534     25325211.0
3  2022-06         35413     22761054.0


In [17]:
q6 = pd.read_sql("""
    SELECT
        size,
        COUNT(*)                 AS total_orders,
        ROUND(SUM(revenue),2)    AS total_revenue
    FROM sales
    GROUP BY SIZE
    ORDER BY total_orders DESC
""", conn)
print("\n=== Q6: REVENUE BY SIZE ===")
print(q6)


=== Q6: REVENUE BY SIZE ===
    size  total_orders  total_revenue
0      M         21291     13419894.0
1      L         20800     12767020.0
2     XL         19725     11999696.0
3    XXL         17066     10350376.0
4      S         15953     10258811.0
5    3XL         14051      8869776.0
6     XS         10296      6848687.0
7    6XL           705       564390.0
8    5XL           526       416337.0
9    4XL           408       328518.0
10  Free           356       208266.0


In [18]:
q7 = pd.read_sql("""
    SELECT 
        CASE WHEN b2b = 1 THEN 'B2B' ELSE 'B2C' END AS customer_type,
        COUNT(*)                                    AS total_orders,
        ROUND(SUM(revenue), 2)                      AS total_revenue, 
        ROUND(AVG(amount), 2)                       AS avg_order_value
    FROM sales
    GROUP BY b2b
""", conn)
print("\n=== Q7: B2B vs B2C ===")
print(q7)


=== Q7: B2B vs B2C ===
  customer_type  total_orders  total_revenue  avg_order_value
0           B2C        120334     75414591.0           648.19
1           B2B           843       617180.0           701.33


In [19]:
q8 = pd.read_sql("""
    SELECT
        fulfilment,
        COUNT(*)                  AS total_orders,
        ROUND(SUM(revenue), 2)    AS total_revenue,
        ROUND(AVG(amount), 2)     AS avg_order_value
    FROM sales
    GROUP BY fulfilment
    ORDER BY total_revenue DESC
""", conn)
print("\n=== Q8: FULFILMENT TYPE ===")
print(q8)


=== Q8: FULFILMENT TYPE ===
  fulfilment  total_orders  total_revenue  avg_order_value
0     Amazon         83636     54711512.0           649.48
1   Merchant         37541     21320259.0           646.51


In [20]:
q9 = pd.read_sql("""
    SELECT
        category,
        COUNT(*)                    AS cancelled_orders
    FROM sales
    WHERE status = 'Cancelled'
    GROUP BY category
    ORDER BY cancelled_orders DESC
""", conn)
print("\n=== Q9: CANCELLATIONS BY CATEGORY ===")
print(q9)


=== Q9: CANCELLATIONS BY CATEGORY ===
        category  cancelled_orders
0            Set              4199
1          kurta              4195
2  Western Dress              1335
3            Top               829
4   Ethnic Dress                80
5         Blouse                75
6         Bottom                41
7          Saree                12


In [22]:
q10 = pd.read_sql("""
    SELECT
        ship_city,
        ship_state,
        COUNT(*)                    AS total_orders,
        ROUND(SUM(revenue), 2)      AS total_revenue
    FROM sales
    GROUP BY ship_city, ship_state
    ORDER BY total_orders DESC
    LIMIT 5
""", conn)
print("\n=== Q10: TOP 5 CITIES BY ORDERS ===")
print(q10)


=== Q10: TOP 5 CITIES BY ORDERS ===
   ship_city   ship_state  total_orders  total_revenue
0  BENGALURU    KARNATAKA         11324      7104012.0
1  HYDERABAD    TELANGANA          8559      5412107.0
2     MUMBAI  MAHARASHTRA          6809      4173450.0
3    CHENNAI   TAMIL NADU          5959      3500826.0
4  NEW DELHI        DELHI          5863      3752570.0


In [23]:
q11 = pd.read_sql("""
    SELECT
        week,
        COUNT(*)                    AS total_orders,
        ROUND(SUM(revenue), 2)      AS total_revenue
    FROM sales
    GROUP BY week
    ORDER BY week
""", conn)
print("\n=== Q11: WEEKLY REVENUE TREND ===")
print(q11)


=== Q11: WEEKLY REVENUE TREND ===
    week  total_orders  total_revenue
0     13          4572      2821513.0
1     14         10584      6542238.0
2     15         11013      6606481.0
3     16         11408      6829038.0
4     17         10358      6203634.0
5     18         11153      6998611.0
6     19          8035      5097398.0
7     20          7833      5011362.0
8     21          8317      5471060.0
9     22          9261      6024474.0
10    23          9807      6305020.0
11    24          8323      5380822.0
12    25          7695      4876221.0
13    26          2818      1863899.0


In [24]:
q12 = pd.read_sql("""
    SELECT
        category,
        COUNT(*)                    AS high_value_orders,
        ROUND(SUM(revenue), 2)      AS total_revenue,
        ROUND(AVG(amount), 2)       AS avg_amount
    FROM sales
    WHERE amount > 1000
    GROUP BY category
    ORDER BY high_value_orders DESC
""", conn)
print("\n=== Q12: HIGH VALUE ORDERS (₹1000+) ===")
print(q12)


=== Q12: HIGH VALUE ORDERS (₹1000+) ===
        category  high_value_orders  total_revenue  avg_amount
0            Set              12219     14775286.0     1221.86
1  Western Dress                747       932688.0     1169.84
2          kurta                265       421400.0     1143.41
3   Ethnic Dress                 58        69968.0     1206.34
4            Top                 29        62405.0     1203.97
5          Saree                  7        17201.0     1420.43
6         Blouse                  5         6399.0     1136.32
7         Bottom                  1            0.0     1028.58


In [25]:
q13 = pd.read_sql("""
    SELECT
        category,
        COUNT(*)                    AS total_orders,
        ROUND(SUM(revenue), 2)      AS total_revenue
    FROM sales
    GROUP BY category
    HAVING total_orders > 10000
    ORDER BY total_orders DESC
""", conn)
print("\n=== Q13: CATEGORIES WITH 10,000+ ORDERS ===")
print(q13)


=== Q13: CATEGORIES WITH 10,000+ ORDERS ===
        category  total_orders  total_revenue
0            Set         47040     37932332.0
1          kurta         46716     20674816.0
2  Western Dress         14704     10707932.0
3            Top         10165      5242931.0


In [26]:
q14 = pd.read_sql("""
    SELECT
        fulfilment,
        category,
        COUNT(*)                    AS total_orders,
        ROUND(SUM(revenue), 2)      AS total_revenue
    FROM sales
    GROUP BY fulfilment, category
    ORDER BY fulfilment, total_revenue DESC
""", conn)
print("\n=== Q14: FULFILMENT + CATEGORY BREAKDOWN ===")
print(q14)


=== Q14: FULFILMENT + CATEGORY BREAKDOWN ===
   fulfilment       category  total_orders  total_revenue
0      Amazon            Set         32976     27823433.0
1      Amazon          kurta         33103     15308925.0
2      Amazon  Western Dress          8040      6300759.0
3      Amazon            Top          7705      4159785.0
4      Amazon   Ethnic Dress           796       581747.0
5      Amazon         Blouse           653       344997.0
6      Amazon          Saree           124       104072.0
7      Amazon         Bottom           236        86879.0
8      Amazon        Dupatta             3          915.0
9    Merchant            Set         14064     10108899.0
10   Merchant          kurta         13613      5365891.0
11   Merchant  Western Dress          6664      4407173.0
12   Merchant            Top          2460      1083146.0
13   Merchant   Ethnic Dress           297       181202.0
14   Merchant         Blouse           228        96262.0
15   Merchant         Bott

In [27]:
q15 = pd.read_sql("""
    SELECT
        ship_state,
        COUNT(*)                    AS cancelled_orders
    FROM sales
    WHERE status = 'Cancelled'
    GROUP BY ship_state
    ORDER BY cancelled_orders DESC
    LIMIT 10
""", conn)
print("\n=== Q15: TOP 10 STATES WITH MOST CANCELLATIONS ===")
print(q15)


=== Q15: TOP 10 STATES WITH MOST CANCELLATIONS ===
       ship_state  cancelled_orders
0     MAHARASHTRA              1802
1       KARNATAKA              1318
2       TELANGANA               948
3      TAMIL NADU               932
4   UTTAR PRADESH               918
5          KERALA               744
6  ANDHRA PRADESH               526
7           DELHI               503
8     WEST BENGAL               473
9         Gujarat               380


In [28]:
conn.close()
print("\n Database connection closed ✓")


 Database connection closed ✓
